In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from slc_rtc_lookup import query_slc_metadata_by_geometry, check_rtc_s1_by_many_slc_ids, check_rtc_s1_by_one_slc_id
from shapely.geometry import box
import pandas as pd
from datetime import datetime, timedelta

In [15]:
la_bbox = box(-118.6682, 33.7037, -118.1553, 34.3373)
bruss_bbox = box(3.67, 49.64, 5.67, 51.64)
start = datetime(2026, 3, 1)
stop = start + timedelta(days=15)

In [16]:
df_slc = query_slc_metadata_by_geometry(geometry=bruss_bbox, 
                                        start_time=start, 
                                        stop_time=stop)
df_slc.head()

,slc_id,orbit_pass,slc_polarizations,track_number,start_time,url,s3_uri,size_gb,geometry
0,S1A_IW_SLC__1SDV_20260315T055108_20260315T0551...,DESCENDING,VV+VH,37,2026-03-15 05:51:08+00:00,https://datapool.asf.alaska.edu/SLC/SA/S1A_IW_...,s3://asf-ngap2w-p-s1-slc-7b420b89/S1A_IW_SLC__...,4.517733,"POLYGON ((6.98306 49.3045, 3.44752 49.70722, 3..."
1,S1A_IW_SLC__1SDV_20260315T055043_20260315T0551...,DESCENDING,VV+VH,37,2026-03-15 05:50:43+00:00,https://datapool.asf.alaska.edu/SLC/SA/S1A_IW_...,s3://asf-ngap2w-p-s1-slc-7b420b89/S1A_IW_SLC__...,4.434777,"POLYGON ((7.45268 50.78846, 3.8026 51.19319, 3..."
2,S1A_IW_SLC__1SDV_20260315T055018_20260315T0550...,DESCENDING,VV+VH,37,2026-03-15 05:50:18+00:00,https://datapool.asf.alaska.edu/SLC/SA/S1A_IW_...,s3://asf-ngap2w-p-s1-slc-7b420b89/S1A_IW_SLC__...,4.401681,"POLYGON ((7.94585 52.27063, 4.16998 52.67824, ..."
3,S1C_IW_SLC__1SDV_20260314T055803_20260314T0558...,DESCENDING,VV+VH,110,2026-03-14 05:58:03+00:00,https://datapool.asf.alaska.edu/SLC/SC/S1C_IW_...,s3://asf-ngap2w-p-s1-slc-7b420b89/S1C_IW_SLC__...,4.492805,"POLYGON ((5.34427 50.51848, 1.68506 50.92685, ..."
4,S1C_IW_SLC__1SDV_20260314T055737_20260314T0558...,DESCENDING,VV+VH,110,2026-03-14 05:57:37+00:00,https://datapool.asf.alaska.edu/SLC/SC/S1C_IW_...,s3://asf-ngap2w-p-s1-slc-7b420b89/S1C_IW_SLC__...,4.181793,"POLYGON ((5.85198 52.05752, 2.06365 52.46868, ..."


In [17]:
slc_ids = df_slc['slc_id'].unique().tolist()
slc_ids[:3], df_slc.shape[0]

(['S1A_IW_SLC__1SDV_20260315T055108_20260315T055136_063634_07FF9D_1FA5',
  'S1A_IW_SLC__1SDV_20260315T055043_20260315T055110_063634_07FF9D_E8AD',
  'S1A_IW_SLC__1SDV_20260315T055018_20260315T055045_063634_07FF9D_D68B'],
 31)

In [18]:
# df_rtc_check_one = check_rtc_s1_by_one_slc_id(slc_ids[0])
# df_rtc_check_one.head()

In [19]:
# df_rtc_check_one['rtc_opera_id'].isnull().sum()

In [ ]:
df_rtc_check = check_rtc_s1_by_many_slc_ids(slc_ids)
df_rtc_check.head()

Checking RTC-S1 for SLCs:   0%|          | 0/31 [00:00<?, ?it/s]

In [ ]:
df_rtc_check_deduped = df_rtc_check.drop_duplicates(subset=['jpl_burst_id']).reset_index(drop=True)
f'Deduped: {df_rtc_check_deduped.shape[0]:,}, Original: {df_rtc_check.shape[0]:,}'

'Deduped: 363, Original: 1,576'

In [ ]:
n_missing_products = df_rtc_check_deduped[df_rtc_check_deduped['rtc_opera_id'].isnull()].shape[0]
total_products_with_rtc = df_rtc_check_deduped.shape[0]

In [ ]:
f'There are {n_missing_products:,} missing RTC S1 products out of {total_products_with_rtc:,} RTC-S1 products'

'There are 0 missing RTC S1 products out of 363 RTC-S1 products'

In [ ]:
slc_ids_with_missing_rtc = df_rtc_check_deduped[df_rtc_check_deduped['rtc_opera_id'].isnull()]['slc_id'].unique().tolist()
total_slc_ids = len(df_rtc_check_deduped['slc_id'].unique().tolist())

In [ ]:
f'There are {len(slc_ids_with_missing_rtc):,} SLC products with missing RTC S1 products out of {total_slc_ids} SLCs produced from {start.strftime("%Y%m%d")} to {stop.strftime("%Y%m%d")}'

'There are 0 SLC products with missing RTC S1 products out of 16 SLCs produced from 20260201 to 20260216'

In [ ]:
df_rtc_check_deduped[df_rtc_check_deduped['rtc_opera_id'].isnull()].to_csv(f'missing_rtc_s1_prods_{start.strftime("%Y%m%d")}__{stop.strftime("%Y%m%d")}.csv')